# Parametric particle-trajectory PINN
One-way analytical Poiseuille flow with linear Stokes drag. The network learns a position correction from acceleration residuals. Start this notebook from the repository root or from the package directory.

In [ ]:
from pathlib import Path
import os

os.chdir("/kaggle/working")
package = Path("parametric_particle_pinn")
package.mkdir(exist_ok=True)
files = {'__init__.py': '"""Direct conditional particle trajectories in analytical Poiseuille flow."""\n', 'config.json': '{\n  "schema": "stokes-poiseuille-particle-v1",\n  "physics": {\n    "length": 2.2, "height": 0.30, "centerline_speed": 1.0,\n    "reynolds": 10.0, "diameter": 0.025, "density_ratio": 1000.0,\n    "drag_law": "stokes", "launch_gap": 0.00001\n  },\n  "domain": {\n    "horizon": 5.0, "vx0": [-0.4, 1.5],\n    "inward_speed": [0.15, 1.20], "interior_vy0": [-1.20, 1.20]\n  },\n  "model": {\n    "time_basis": "chebyshev", "degree": 24, "dtype": "float64", "time_feature_scale": 0.25\n  },\n  "training": {\n    "seed": 42, "dev_seed": 1201, "final_seed": 9327, "threads": 2,\n    "horizontal_adam_steps": 3000, "transverse_adam_steps": 1600,\n    "transverse_adam_rate": 0.008, "batch_size": 384, "horizontal_adam_rate": 0.004,\n    "learning_rate_min": 0.00015, "check_every": 250,\n    "lbfgs_steps": 500, "lbfgs_bank_size": 4096, "lbfgs_block_steps": 50,\n    "residual_velocity_scale": [1.0, 1.2]\n  },\n  "validation": {\n    "samples_per_case": 121, "event_scan_step": 0.02,\n    "absolute_position_max": [0.004, 0.0015],\n    "absolute_velocity_max": [0.012, 0.006],\n    "scaled_residual_rms": [0.025, 0.020],\n    "scaled_residual_max": [0.10, 0.08],\n    "initial_max": 1e-12, "event_time_max": 0.04,\n    "reference_check_max": 1e-9\n  }\n}\n', 'domain.py': '"""Release validation and independent banks of training/evaluation conditions."""\n\nimport numpy as np\n\nfrom .model import input_bounds\n\n\ndef wall_release(config, side, x0, vx0, inward_speed):\n    """Build Cartesian release inputs from a lower or upper wall launcher."""\n    if side not in ("lower", "upper"):\n        raise ValueError("side must be \'lower\' or \'upper\'.")\n    x0, vx0, speed = np.broadcast_arrays(x0, vx0, inward_speed)\n    low, high = config["domain"]["inward_speed"]\n    if not np.isfinite(speed).all() or np.any((speed < low) | (speed > high)):\n        raise ValueError(f"inward_speed must be in [{low}, {high}].")\n    p = config["physics"]\n    y0 = p["diameter"] / 2 + p["launch_gap"]\n    if side == "upper":\n        y0, speed = p["height"] - y0, -speed\n    releases = np.stack((x0, np.full(x0.shape, y0), vx0, speed), axis=-1)\n    check_releases(config, releases)\n    return releases\n\n\ndef check_releases(config, releases):\n    releases = np.asarray(releases, dtype=float)\n    if releases.shape[-1:] != (4,) or not releases.size or not np.isfinite(releases).all():\n        raise ValueError("Provide finite release conditions with last axis [x0, y0, vx0, vy0].")\n    lower, upper = np.array(input_bounds(config)[1:]).T\n    if np.any((releases < lower) | (releases > upper)):\n        raise ValueError("Initial condition is outside the supported Cartesian bounds.")\n\n\ndef sample_releases(config, rng, count, *, boundaries=False):\n    lower, upper = np.array(input_bounds(config)[1:]).T\n    releases = rng.uniform(lower, upper, size=(count, 4))\n    # Include both exact Master3 launcher heights as well as the continuous interior.\n    groups = np.arange(count) % 4\n    for group, side in ((0, "lower"), (1, "upper")):\n        subset = groups == group\n        speed = rng.uniform(*config["domain"]["inward_speed"], size=int(subset.sum()))\n        releases[subset] = wall_release(config, side, releases[subset, 0],\n                                       releases[subset, 2], speed)\n    releases[groups == 2, 3] = 0\n    if boundaries:\n        faces = rng.permutation(count)[:count // 4]\n        for row in faces:\n            edge = rng.integers(0, 2)\n            if groups[row] < 2:\n                axis = rng.choice((0, 2, 3))\n                if axis == 3:\n                    speed = config["domain"]["inward_speed"][edge]\n                    releases[row, 3] = speed if groups[row] == 0 else -speed\n                    continue\n            else:\n                axis = rng.integers(0, 4)\n            releases[row, axis] = (lower, upper)[edge][axis]\n    return releases\n', 'inference.py': '"""Batched trajectory inference and scalar root finding on the neural path."""\n\nimport numpy as np\nimport torch\nfrom scipy.optimize import brentq\n\nfrom .domain import check_releases, wall_release  # public standalone adapter\nfrom .model import ParticlePINN, load_checkpoint, particle_state\n\nEVENTS = ("lower_wall", "upper_wall", "inlet_exit", "outlet")\nAXES = (1, 1, 0, 0)\n\n\ndef _evaluate(model, releases, times, *, acceleration=True):\n    batch, count = times.shape\n    shapes = (batch, count, 2)\n    time = model.bounds.new_tensor(times.reshape(-1, 1))\n    release = model.bounds.new_tensor(releases).repeat_interleave(count, dim=0)\n    values = [[], [], []] if acceleration else [[], []]\n    for start in range(0, len(time), 4096):\n        stop = start + 4096\n        state = particle_state(model, time[start:stop], release[start:stop],\n                               acceleration=acceleration)\n        for component, value in zip(values, state):\n            component.append(value.detach().cpu().numpy())\n    return tuple(np.concatenate(component).reshape(shapes) for component in values)\n\n\ndef _events(model, releases):\n    config = model.config\n    p, d = config["physics"], config["domain"]\n    horizon = d["horizon"]\n    radius = p["diameter"] / 2\n    offsets = np.array([radius, p["height"] - radius, 0.0, p["length"]])\n    signs = np.array([1, -1, 1, -1])\n    samples = int(np.ceil(horizon / config["validation"]["event_scan_step"]))\n    mesh = np.linspace(0, horizon, samples + 1)\n    positions, velocities = _evaluate(model, releases,\n                                      np.broadcast_to(mesh, (len(releases), len(mesh))),\n                                      acceleration=False)\n    if not np.isfinite(positions).all() or not np.isfinite(velocities).all():\n        raise FloatingPointError("Non-finite neural trajectory during event scan.")\n    distances = (positions[..., AXES] - offsets) * signs\n    rates = velocities[..., AXES] * signs\n    event_types, event_times = [], []\n    for index, release in enumerate(releases):\n        initial_rate = rates[index, 0]\n        first_type, first_time = "time_limit", horizon\n        for face, event in enumerate(EVENTS):\n            values, slopes = distances[index, :, face], rates[index, :, face]\n            if values[0] == 0 and (initial_rate[face] < 0 or\n                                   (face < 2 and initial_rate[face] == 0)):\n                first_type, first_time = event, 0.0\n                break\n\n            axis, offset, sign = AXES[face], offsets[face], signs[face]\n\n            def scalar(time, velocity=False):\n                t = model.bounds.new_tensor([[time]])\n                r = model.bounds.new_tensor(release[None, :])\n                if velocity:\n                    _, v, _ = particle_state(model, t, r, acceleration=False)\n                    return sign * v[0, axis].item()\n                with torch.no_grad():\n                    return sign * (model(t, r)[0, axis].item() - offset)\n\n            # Inspect sign changes and minima for brief excursions and tangencies.\n            crossing = (values[:-1] >= 0) & (values[1:] < 0)\n            endpoint = (values[-1] == 0 and slopes[-1] < 0)\n            minimum = (slopes[:-1] < 0) & (slopes[1:] >= 0)\n            for left in np.flatnonzero(crossing | minimum):\n                start, stop = mesh[left:left + 2]\n                if start >= first_time:\n                    break\n                if crossing[left]:\n                    hit = brentq(scalar, start, stop, xtol=1e-12)\n                else:\n                    bottom = brentq(lambda t: scalar(t, velocity=True), start, stop, xtol=1e-12)\n                    clearance = scalar(bottom)\n                    if face < 2 and abs(clearance) <= 1e-12:\n                        hit = bottom\n                    elif clearance < 0 and values[left] > 0:\n                        hit = brentq(scalar, start, bottom, xtol=1e-12)\n                    else:\n                        continue\n                if hit < first_time:\n                    first_type, first_time = event, hit\n                break\n            if endpoint and first_type == "time_limit":\n                first_type = event\n        event_types.append(first_type)\n        event_times.append(first_time)\n    return np.array(event_types), np.array(event_times)\n\n\ndef predict_trajectory(checkpoint, x0, y0, vx0, vy0, times, *, device="cpu"):\n    """Predict at shared 1-D times or one array per broadcast release condition.\n\n    Returns nondimensional time, position, velocity, acceleration, an event\n    dictionary and validity mask. Trajectory state is NaN beyond the first event.\n    Pass an already loaded model to reuse it across calls.\n    """\n    model = checkpoint if isinstance(checkpoint, ParticlePINN) else load_checkpoint(checkpoint, device)\n    releases = np.stack(np.broadcast_arrays(x0, y0, vx0, vy0), axis=-1).astype(float)\n    check_releases(model.config, releases)\n    batch_shape = releases.shape[:-1]\n    releases = releases.reshape(-1, 4)\n    times = np.atleast_1d(np.asarray(times, dtype=float))\n    if not times.size or not np.isfinite(times).all():\n        raise ValueError("Provide finite nonempty elapsed times.")\n    horizon = model.config["domain"]["horizon"]\n    if np.any((times < 0) | (times > horizon)):\n        raise ValueError(f"Elapsed times must be in [0, {horizon}].")\n    output_shape = (*batch_shape, times.shape[-1])\n    if times.ndim != 1 and times.shape != output_shape:\n        raise ValueError("Use shared 1-D times or match the release batch axes.")\n    elapsed = np.broadcast_to(times, output_shape)\n    flat_times = elapsed.reshape(len(releases), -1)\n    kind, end = _events(model, releases)\n    valid = flat_times <= end[:, None]\n    # Evaluate invalid queries at t=0 only, then mask all state fields.\n    state = _evaluate(model, releases, np.where(valid, flat_times, 0.0))\n    result = {"elapsed_time": elapsed, "valid": valid.reshape(output_shape),\n              "event": {"type": kind.reshape(batch_shape), "time": end.reshape(batch_shape)}}\n    if not batch_shape:\n        result["event"] = {"type": kind[0].item(), "time": end[0].item()}\n    for name, values in zip(("position", "velocity", "acceleration"), state):\n        values[~valid] = np.nan\n        result[name] = values.reshape(*output_shape, 2)\n    return result\n', 'model.py': '"""Hard initial conditions and automatic derivatives in nondimensional time."""\n\nimport json\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\n\nCONFIG_PATH = Path(__file__).with_name("config.json")\nRELEASE_INPUTS = ("x0", "y0", "vx0", "vy0")\n\n\ndef read_config(path=CONFIG_PATH):\n    return json.loads(Path(path).read_text())\n\n\ndef relaxation_time(config):\n    p = config["physics"]\n    return p["density_ratio"] * p["reynolds"] * p["diameter"] ** 2 / 18\n\n\ndef input_bounds(config):\n    p, d = config["physics"], config["domain"]\n    radius = p["diameter"] / 2\n    return [\n        [0, d["horizon"]], [0, p["length"]], [radius, p["height"] - radius],\n        d["vx0"], d["interior_vy0"],\n    ]\n\n\nclass ParticlePINN(nn.Module):\n    """A conditional correction with learned time coefficients and release features."""\n\n    def __init__(self, config):\n        super().__init__()\n        self.config = config\n        p, architecture = config["physics"], config["model"]\n        if p["drag_law"] != "stokes":\n            raise ValueError("ParticlePINN requires linear Stokes drag.")\n        dtype = getattr(torch, architecture["dtype"])\n        self.register_buffer("bounds", torch.tensor(input_bounds(config), dtype=dtype))\n        degree = architecture["degree"]\n        self.register_buffer("frequencies", torch.arange(degree + 1, dtype=dtype))\n        self.register_buffer("basis_transform", torch.eye(degree + 1, dtype=dtype))\n        self.horizontal = nn.Linear(degree + 1, 4, bias=False, dtype=dtype)\n        self.transverse = nn.Linear(degree + 1, 1, bias=False, dtype=dtype)\n        nn.init.zeros_(self.horizontal.weight)\n        nn.init.zeros_(self.transverse.weight)\n\n    def _time_features(self, time):\n        # Logarithmic normalization resolves rapid early transients. The\n        # endpoint inset keeps acos differentiable at t=0 and the horizon.\n        horizon = self.config["domain"]["horizon"]\n        scale = self.config["model"]["time_feature_scale"]\n        shaped = 2 * torch.log1p(time / scale) / torch.log1p(time.new_tensor(horizon / scale)) - 1\n        basis = torch.cos(torch.acos(0.999 * shaped) * self.frequencies)\n        return time.square() / (time + scale) * basis\n\n    def precondition(self):\n        """Whiten the time basis in the Stokes differential operator norm.\n\n        This uses the differential operator and time grid only. No release or\n        reference trajectory values enter the transform.\n        """\n        horizon = self.config["domain"]["horizon"]\n        grid = torch.linspace(0, horizon, 513, dtype=self.bounds.dtype, device=self.bounds.device)\n        nodes = torch.cat((grid, horizon * (grid / horizon).pow(3)))\n        features = lambda t: self._time_features(t.view(1))\n        first = torch.func.jacfwd(features)\n        second = torch.func.jacfwd(first)\n        operator = torch.func.vmap(first)(nodes) + relaxation_time(self.config) * torch.func.vmap(second)(nodes)\n        _, upper = torch.linalg.qr(operator / len(nodes)**0.5, mode="reduced")\n        with torch.no_grad():\n            self.basis_transform.copy_(torch.linalg.solve_triangular(\n                upper, torch.eye(upper.shape[0], device=upper.device, dtype=upper.dtype), upper=True))\n\n    def forward(self, time, release):\n        # Inputs keep their physical scale on the autograd graph.\n        time_basis = self._time_features(time) @ self.basis_transform\n        a, b, c, d = self.horizontal(time_basis).unbind(dim=-1)\n        e = self.transverse(time_basis).squeeze(-1)\n        p = self.config["physics"]\n        height_coordinate = 2 * release[..., 1] / p["height"] - 1\n        vx = release[..., 2] / p["centerline_speed"]\n        vy = release[..., 3] / self.bounds[4, 1]\n        correction = torch.stack((\n            a * (1 - height_coordinate.square()) + b * height_coordinate * vy\n            + c * vy.square() + d * vx,\n            e * vy,\n        ), dim=-1)\n        return release[..., :2] + time * release[..., 2:] + correction\n\n\ndef particle_state(model, time, release, *, train=False, acceleration=True):\n    """Return positions and first and optionally second elapsed-time derivatives."""\n    with torch.enable_grad():\n        time = time.detach().requires_grad_(True)\n        position = model(time, release)\n        velocity = [\n            torch.autograd.grad(component.sum(), time, create_graph=train or acceleration,\n                                retain_graph=train or acceleration or axis == 0)[0]\n            for axis, component in enumerate(position.unbind(dim=-1))\n        ]\n        if acceleration:\n            second = torch.cat([\n                torch.autograd.grad(component.sum(), time, create_graph=train,\n                                    retain_graph=train or axis == 0)[0]\n                for axis, component in enumerate(velocity)\n            ], dim=-1)\n        else:\n            second = None\n        return position, torch.cat(velocity, dim=-1), second\n\n\ndef scaled_residual(config, position, velocity, acceleration):\n    p = config["physics"]\n    eta = position[..., 1] / p["height"]\n    carrier = torch.stack((4 * p["centerline_speed"] * eta * (1 - eta),\n                           torch.zeros_like(eta)), dim=-1)\n    scale = velocity.new_tensor(config["training"]["residual_velocity_scale"])\n    return (relaxation_time(config) * acceleration + velocity - carrier) / scale\n\n\ndef load_checkpoint(path, device="cpu"):\n    data = torch.load(path, map_location=device, weights_only=True)\n    config = data["config"]\n    if config["schema"] != read_config()["schema"]:\n        raise ValueError("Unsupported particle checkpoint schema.")\n    model = ParticlePINN(config).to(device).eval()\n    model.load_state_dict(data["model_state"])\n    model.requires_grad_(False)\n    return model\n', 'train.py': '"""Physics-only Adam and fixed-bank L-BFGS training."""\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport numpy as np\nimport torch\n\nfrom .domain import sample_releases\nfrom .model import CONFIG_PATH, ParticlePINN, particle_state, read_config, scaled_residual\n\n\ndef _bank(model, rng, count):\n    config = model.config\n    releases = sample_releases(config, rng, count, boundaries=True)\n    units = rng.random(count)\n    units[::4] **= 3\n    units[1::24] = 0\n    units[2::24] = 1\n    return (model.bounds.new_tensor(units[:, None] * config["domain"]["horizon"]),\n            model.bounds.new_tensor(releases))\n\n\ndef _loss(model, bank, *, train):\n    state = particle_state(model, *bank, train=train)\n    return scaled_residual(model.config, *state).square().mean(dim=0)\n\n\ndef train(config, output, device="cpu"):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    (output / "config.json").write_text(json.dumps(config, indent=2) + "\\n")\n    settings = config["training"]\n    torch.manual_seed(settings["seed"])\n    torch.set_num_threads(settings["threads"])\n    rng = np.random.default_rng(settings["seed"])\n    model = ParticlePINN(config).to(device)\n    model.precondition()\n    dev = _bank(model, np.random.default_rng(settings["dev_seed"]), 1536)\n    metadata = {\n        "torch": str(torch.__version__), "device": str(device), "optimizer": ["Adam"],\n        "normalization": model.bounds.tolist(),\n        "initial_ansatz": "q0 + t*v0 + t^2/(t+s)*learned_correction",\n    }\n    best = float("inf")\n    best_step = 0\n    clock = perf_counter()\n\n    def save(filename, step, score):\n        data = {"config": config, "model_state": model.state_dict(), "step": step,\n                "dev_scaled_mse": score, "training": metadata.copy()}\n        temp = output / (filename + ".tmp")\n        torch.save(data, temp)\n        temp.replace(output / filename)\n\n    def check(phase, step, train_components):\n        nonlocal best, best_step\n        dev_components = _loss(model, dev, train=False).detach().tolist()\n        score = sum(dev_components)\n        if not np.isfinite(score):\n            raise FloatingPointError("Non-finite development loss.")\n        row = dict(phase=phase, step=step, train_scaled_mse=train_components,\n                   dev_scaled_mse=dev_components, elapsed_seconds=perf_counter() - clock)\n        with (output / "training.jsonl").open("a") as log:\n            log.write(json.dumps(row) + "\\n")\n        print(json.dumps(row), flush=True)\n        if score < best:\n            best, best_step = score, step\n            save("best.pt", step, dev_components)\n        return dev_components\n\n    components = check("initial", 0, None)\n    step = 0\n    for phase, phase_steps, rate in (\n        ("adam_transverse", settings["transverse_adam_steps"], settings["transverse_adam_rate"]),\n        ("adam_horizontal", settings["horizontal_adam_steps"], settings["horizontal_adam_rate"]),\n    ):\n        component = 1 if phase == "adam_transverse" else 0\n        head = model.transverse if component == 1 else model.horizontal\n        adam = torch.optim.Adam(head.parameters(), lr=rate)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(\n            adam, max(phase_steps, 1), eta_min=settings["learning_rate_min"])\n        for phase_step in range(1, phase_steps + 1):\n            model.zero_grad(set_to_none=True)\n            losses = _loss(model, _bank(model, rng, settings["batch_size"]), train=True)\n            losses[component].backward()\n            adam.step()\n            scheduler.step()\n            step += 1\n            if phase_step % settings["check_every"] == 0 or phase_step == phase_steps:\n                components = check(phase, step, losses.detach().tolist())\n\n    total_refinement = settings["lbfgs_steps"]\n    if total_refinement:\n        metadata["optimizer"].append("horizontal L-BFGS (strong Wolfe)")\n        fixed_bank = _bank(model, rng, settings["lbfgs_bank_size"])\n        block = settings["lbfgs_block_steps"]\n        lbfgs = torch.optim.LBFGS(model.horizontal.parameters(), max_iter=block, max_eval=2 * block,\n                                  history_size=50, line_search_fn="strong_wolfe",\n                                  tolerance_grad=1e-12, tolerance_change=1e-14)\n\n        def closure():\n            model.zero_grad(set_to_none=True)\n            loss = _loss(model, fixed_bank, train=True)[0]\n            loss.backward()\n            return loss\n\n        adam_total = step\n        for start in range(0, total_refinement, block):\n            lbfgs.param_groups[0]["max_iter"] = min(block, total_refinement - start)\n            previous_step = step\n            lbfgs.step(closure)\n            state = lbfgs.state[model.horizontal.weight]\n            refinement = state["n_iter"]\n            step = adam_total + refinement\n            metadata["lbfgs_iterations"] = refinement\n            metadata["lbfgs_closure_evaluations"] = state["func_evals"]\n            components = check("lbfgs", step, _loss(model, fixed_bank, train=False).tolist())\n            if step == previous_step:\n                break\n    save("final.pt", step, components)\n    summary = {"training_completed": True, "best_step": best_step, "final_step": step,\n               "best_dev_total_scaled_mse": best, "elapsed_seconds": perf_counter() - clock,\n               **metadata}\n    (output / "training_summary.json").write_text(json.dumps(summary, indent=2) + "\\n")\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, default=CONFIG_PATH)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--device", default="cpu")\n    args = parser.parse_args()\n    train(read_config(args.config), args.output, args.device)\n\n\nif __name__ == "__main__":\n    main()\n', 'validate.py': '"""Independent Stokes ODE validation and time-matched neural comparisons."""\n\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport numpy as np\nimport torch\nfrom scipy.integrate import solve_ivp\n\nfrom .domain import sample_releases, wall_release\nfrom .inference import predict_trajectory\nfrom .model import load_checkpoint\n\n\ndef reference(config, release):\n    p = config["physics"]\n    height, length = p["height"], p["length"]\n    radius = p["diameter"] / 2\n    tau = p["density_ratio"] * p["reynolds"] * p["diameter"] ** 2 / 18\n    horizon = config["domain"]["horizon"]\n\n    def rhs(_time, state):\n        y, vx, vy = state[1:]\n        eta = y / height\n        return vx, vy, (4 * p["centerline_speed"] * eta * (1 - eta) - vx) / tau, -vy / tau\n\n    def lower(_time, state):\n        return state[1] - radius\n\n    def upper(_time, state):\n        return height - radius - state[1]\n\n    def inlet(_time, state):\n        return state[0]\n\n    def outlet(_time, state):\n        return length - state[0]\n\n    for name, clearance, rate in (\n        ("lower_wall", lower(0, release), release[3]),\n        ("upper_wall", upper(0, release), -release[3]),\n        ("inlet_exit", inlet(0, release), release[2]),\n        ("outlet", outlet(0, release), -release[2]),\n    ):\n        if clearance == 0 and (rate < 0 or (rate == 0 and name.endswith("wall"))):\n            return name, 0.0, lambda t: np.broadcast_to(release, (np.size(t), 4))\n\n    events = [lower, upper, inlet, outlet]\n    for event in events:\n        event.terminal, event.direction = True, -1\n    sol = solve_ivp(rhs, (0, horizon), release, method="DOP853", rtol=2e-12, atol=2e-14,\n                    max_step=0.05, dense_output=True, events=events)\n    if not sol.success:\n        raise RuntimeError(sol.message)\n    names = ("lower_wall", "upper_wall", "inlet_exit", "outlet")\n    kind = next((name for name, times in zip(names, sol.t_events) if len(times)), "time_limit")\n    return kind, float(sol.t[-1]), lambda t: sol.sol(t).T\n\n\ndef conditions(config, split):\n    seed = config["training"]["final_seed" if split == "final" else "dev_seed"]\n    # A separate, fixed RNG stream also separates diagnostic dev trajectories\n    # from the residual-only development bank used to select best.pt.\n    rng = np.random.default_rng(np.random.SeedSequence([seed, 1]))\n    if split == "dev":\n        return [(f"dev_{i:02d}", row) for i, row in\n                enumerate(sample_releases(config, rng, 32, boundaries=True))]\n    p, d = config["physics"], config["domain"]\n    height, length = p["height"], p["length"]\n    radius = p["diameter"] / 2\n    vmin, vmax = d["vx0"]\n    smin, smax = d["inward_speed"]\n    cases = [\n        ("lower_wall_slow", wall_release(config, "lower", 0.315, vmin, smin)),\n        ("lower_wall_fast", wall_release(config, "lower", 0.55, vmax, smax)),\n        ("upper_wall_slow", wall_release(config, "upper", 0.72, 0.02, smin)),\n        ("upper_wall_fast", wall_release(config, "upper", 1.10, -0.22, smax)),\n        ("lower_contact", (0.9, 0.048, -0.15, -0.60)),\n        ("upper_contact", (0.8, 0.25, 0.55, 0.47)),\n        ("near_lower_contact", (1.1, radius + 1e-5, 0.7, -1.1)),\n        ("near_upper_contact", (1.1, height - radius - 1e-5, 0.7, 1.1)),\n        ("inlet_exit", (0.022, 0.13, vmin, 0.0)),\n        ("outlet_exit", (length - 0.08, 0.21, vmax, 0.12)),\n        ("inlet_edge_inward", (0.0, 0.15, 0.45, 0.0)),\n        ("outlet_edge_inward", (length, 0.085, -0.22, -0.08)),\n        ("inlet_edge_outward", (0.0, 0.18, vmin, 0.0)),\n        ("outlet_edge_outward", (length, 0.23, vmax, 0.0)),\n        ("lower_center_edge", (0.22, radius, vmax, smin)),\n        ("upper_center_edge", (0.19, height - radius, 0.0, -smin)),\n        ("lower_center_contact_at_release", (0.63, radius, 0.57, 0.0)),\n        ("upper_center_contact_at_release", (0.91, height - radius, -0.11, 0.0)),\n        ("slow_time_limit", (0.36, 0.022, -0.05, 0.0)),\n        ("mid_height_zero_vy", (0.21, height / 2, 0.13, 0.0)),\n        ("interior_rising", (0.31, 0.11, -0.28, 0.23)),\n        ("interior_falling", (0.47, 0.21, 0.96, -0.42)),\n    ]\n    cases.extend((f"unseen_{i:02d}", row) for i, row in\n                 enumerate(sample_releases(config, rng, 48)))\n    return cases\n\n\ndef _plots(output, names, times, neural, truth):\n    import matplotlib.pyplot as plt\n\n    selected = [0, 2, 4, 5, 8, 9, 18, 20] if len(names) > 20 else list(range(len(names)))\n    fig, axes = plt.subplots(2, 4, figsize=(13, 6), layout="constrained")\n    for ax, index in zip(axes.flat, selected):\n        ref, pred = truth[index], neural["position"][index]\n        ax.plot(ref[:, 0], ref[:, 1], color="#193856", label="DOP853", linewidth=2)\n        ax.plot(pred[:, 0], pred[:, 1], "--", color="#d24b30", label="PINN")\n        ax.set(title=names[index].replace("_", " "), xlabel="x", ylabel="y")\n        ax.ticklabel_format(useOffset=False)\n    axes.flat[0].legend()\n    fig.savefig(output / "trajectories.png", dpi=170)\n    plt.close(fig)\n\n    fig, axes = plt.subplots(2, 2, figsize=(10, 6), layout="constrained")\n    values = np.concatenate((neural["position"], neural["velocity"]), axis=-1)\n    for component, (ax, label) in enumerate(zip(axes.flat, ("x", "y", "vx", "vy"))):\n        for time, errors in zip(times, abs(values[..., component] - truth[..., component])):\n            ax.plot(time, errors, color="#1d678a", alpha=0.15)\n        ax.set(xlabel="elapsed time", ylabel=f"|{label} error|", ylim=(0, None))\n        ax.set_yscale("symlog", linthresh=1e-7)\n    fig.savefig(output / "errors.png", dpi=170)\n    plt.close(fig)\n\n\ndef validate(checkpoint, output, split="final"):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    model = load_checkpoint(checkpoint)\n    config = model.config\n    torch.set_num_threads(config["training"]["threads"])\n    p, tol = config["physics"], config["validation"]\n    names, releases = zip(*conditions(config, split))\n    releases = np.array(releases)\n    tau = p["density_ratio"] * p["reynolds"] * p["diameter"]**2 / 18\n    unit = np.linspace(0, 1, tol["samples_per_case"])\n    # One warm-up run, then compare inference and numerical integration on the\n    # same releases and same physical query grids, including event detection.\n    predict_trajectory(model, *releases[0], [0.0])\n    refs = [reference(config, row) for row in releases]\n    reference_ends = np.array([item[1] for item in refs])\n    reference_times = reference_ends[:, None] * unit\n    start = perf_counter()\n    benchmark = predict_trajectory(model, *releases.T, reference_times)\n    neural_seconds = perf_counter() - start\n    start = perf_counter()\n    refs = [reference(config, row) for row in releases]\n    for item, times in zip(refs, reference_times):\n        item[2](times)\n    ode_seconds = perf_counter() - start\n\n    ends = np.minimum(reference_ends, benchmark["event"]["time"])\n    times = ends[:, None] * unit\n    neural = predict_trajectory(model, *releases.T, times)\n    assert neural["valid"].all()\n    truth = np.stack([item[2](t) for item, t in zip(refs, times)])\n    position, velocity = neural["position"], neural["velocity"]\n    position_error = abs(position - truth[..., :2])\n    velocity_error = abs(velocity - truth[..., 2:])\n    decay = np.exp(-times / tau)\n    exact_vy = releases[:, 3, None] * decay\n    exact_y = releases[:, 1, None] - tau * releases[:, 3, None] * np.expm1(-times / tau)\n    transverse_errors = abs(np.stack((position[..., 1] - exact_y, velocity[..., 1] - exact_vy), axis=-1))\n    independent_error = abs(np.stack((truth[..., 1] - exact_y, truth[..., 3] - exact_vy), axis=-1))\n    eta = position[..., 1] / p["height"]\n    carrier = np.stack((4 * p["centerline_speed"] * eta * (1 - eta), np.zeros_like(eta)), axis=-1)\n    residual = ((tau * neural["acceleration"] + velocity - carrier) /\n                config["training"]["residual_velocity_scale"])\n    initial_error = abs(np.concatenate((position[:, 0], velocity[:, 0]), axis=-1) - releases)\n\n    rows = []\n    for i, (name, ref) in enumerate(zip(names, refs)):\n        error_q, error_v = position_error[i].max(axis=0), velocity_error[i].max(axis=0)\n        rms_q = np.sqrt((position_error[i]**2).mean(axis=0))\n        rms_v = np.sqrt((velocity_error[i]**2).mean(axis=0))\n        rms_r = np.sqrt((residual[i]**2).mean(axis=0))\n        peak_r = abs(residual[i]).max(axis=0)\n        events_agree = bool(neural["event"]["type"][i] == ref[0])\n        error_t = abs(float(neural["event"]["time"][i]) - ref[1])\n        passed = bool(\n            events_agree and error_t <= tol["event_time_max"] and\n            np.all(error_q <= tol["absolute_position_max"]) and\n            np.all(error_v <= tol["absolute_velocity_max"]) and\n            np.all(rms_r <= tol["scaled_residual_rms"]) and\n            np.all(peak_r <= tol["scaled_residual_max"]) and\n            initial_error[i].max() <= tol["initial_max"] and\n            independent_error[i].max() <= tol["reference_check_max"])\n        rows.append(dict(\n            case=name, **dict(zip(("x0", "y0", "vx0", "vy0"), releases[i])),\n            reference_event=ref[0], neural_event=neural["event"]["type"][i],\n            reference_end=ref[1], neural_end=neural["event"]["time"][i],\n            common_end=ends[i], event_time_error=error_t, event_agrees=events_agree,\n            max_x_error=error_q[0], max_y_error=error_q[1],\n            max_vx_error=error_v[0], max_vy_error=error_v[1],\n            rms_x_error=rms_q[0], rms_y_error=rms_q[1],\n            rms_vx_error=rms_v[0], rms_vy_error=rms_v[1],\n            rms_scaled_rx=rms_r[0], rms_scaled_ry=rms_r[1],\n            max_scaled_rx=peak_r[0], max_scaled_ry=peak_r[1],\n            max_initial_error=initial_error[i].max(),\n            max_exact_y_error=transverse_errors[i, :, 0].max(),\n            max_exact_vy_error=transverse_errors[i, :, 1].max(),\n            max_reference_transverse_error=independent_error[i].max(), passed=passed))\n    fields = list(rows[0])\n    with (output / "cases.csv").open("w", newline="") as handle:\n        writer = csv.DictWriter(handle, fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    coverage = sorted({item[0] for item in refs})\n    summary = dict(\n        split=split, checkpoint=str(checkpoint), case_count=len(rows),\n        passed_count=sum(row["passed"] for row in rows),\n        accepted=all(row["passed"] for row in rows), reference_event_coverage=coverage,\n        failed_cases=[row["case"] for row in rows if not row["passed"]],\n        worst_max_abs_position=position_error.max(axis=(0, 1)).tolist(),\n        worst_max_abs_velocity=velocity_error.max(axis=(0, 1)).tolist(),\n        worst_rms_scaled_residual=np.sqrt((residual**2).mean(axis=1)).max(axis=0).tolist(),\n        worst_max_scaled_residual=abs(residual).max(axis=(0, 1)).tolist(),\n        worst_event_time_error=max(row["event_time_error"] for row in rows),\n        matching_events=sum(row["event_agrees"] for row in rows),\n        worst_initial_error=float(initial_error.max()),\n        worst_independent_transverse_error=float(independent_error.max()),\n        neural_batch_seconds=neural_seconds, independent_ode_seconds=ode_seconds,\n        ode_over_neural=ode_seconds/neural_seconds, thresholds=tol,\n    )\n    (output / "summary.json").write_text(json.dumps(summary, indent=2) + "\\n")\n    _plots(output, names, times, neural, truth)\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("checkpoint", type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--split", choices=("dev", "final"), default="final")\n    args = parser.parse_args()\n    summary = validate(args.checkpoint, args.output, args.split)\n    print(json.dumps(summary, indent=2))\n    raise SystemExit(0 if summary["accepted"] else 1)\n\n\nif __name__ == "__main__":\n    main()\n'}
for filename, text in files.items():
    (package / filename).write_text(text)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "parametric_particle_pinn":
    root = root.parent
sys.path.insert(0, str(root))
from parametric_particle_pinn.model import read_config, load_checkpoint, relaxation_time
from parametric_particle_pinn.train import train

config = read_config()
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = root / "particle-pinn-runs" / stamp
print("Stokes relaxation time:", relaxation_time(config))
print("Output:", run_dir)

## Train using particle physics
Adam fits the transverse head and then the horizontal head. Horizontal L-BFGS reuses one fixed collocation bank. Development residuals select the best checkpoint.

In [ ]:
training_summary = train(config, run_dir, device="cpu")
training_summary

## Load one checkpoint and predict several releases
This cell needs only the saved model and inference API. The releases below are independent of the training and validation banks. Query positions, velocities and accelerations before the predicted stop event.

In [ ]:
import numpy as np
from parametric_particle_pinn.inference import predict_trajectory, wall_release

checkpoint = run_dir / "best.pt"
model = load_checkpoint(checkpoint)
releases = np.vstack((
    wall_release(config, "lower", 0.439, -0.18, 0.41),
    wall_release(config, "upper", 0.751, 0.83, 0.27),
    [0.684, 0.12, 0.29, 0.0],
    [1.531, 0.228, -0.17, 0.65],
))
times = np.linspace(0, config["domain"]["horizon"], 101)
prediction = predict_trajectory(model, *releases.T, times)
for release, event, stop, mask in zip(releases, prediction["event"]["type"],
                                       prediction["event"]["time"], prediction["valid"]):
    print(release, event, round(stop, 6), "valid samples:", int(mask.sum()))

## Independent final validation
The ODE is used only here. Accuracy and event criteria were fixed in the configuration before final validation.

In [ ]:
from parametric_particle_pinn.validate import validate

report = validate(checkpoint, run_dir / "validation", split="final")
print("Accepted:", report["passed_count"], "/", report["case_count"])
from IPython.display import Image, display
for image in ("trajectories.png", "errors.png"):
    display(Image(filename=str(run_dir / "validation" / image)))
assert report["accepted"], report["failed_cases"]